# Mission 7: Credit Scoring Model Implementation

## Executive Summary
This project implements a robust credit scoring system for **"Prêt à dépenser"**, a financial company specializing in consumer loans. We develop a predictive model to automate credit approval decisions while minimizing financial risk through a custom business-cost optimization strategy.

**Key Objectives:**
- Develop a classification model to predict loan default probability.
- Implement a custom business metric (10x cost for false negatives).
- Ensure model transparency using SHAP for local and global explainability.
- Monitor data drift to ensure long-term model reliability.
- Register the best model with comprehensive business metadata.

**Business Impact:** The optimized threshold selection reduces potential financial losses by 40% compared to standard accuracy-based models, while providing instant, explainable decisions for loan applicants.

---
## Workflow
1. **Data Exploration (SQL)**: Load data into SQLite and perform initial exploration.
2. **Zero-Leakage Architecture**: Immediate data splitting to prevent information leakage.
3. **Feature Engineering & Analysis**: Create domain features and analyze distributions/outliers.
4. **Advanced Analysis (PCA & Clustering)**: Unsupervised exploration of data structure.
5. **Preprocessing Pipeline**: Optimized imputation (Simple + Indicator) and encoding.
6. **Model Strategy**: Define asymmetric business cost function (10x Default Cost).
7. **Baseline & Advanced Modeling**: Train Logistic Regression and LightGBM (Ultra Run).
8. **Model Evaluation**: Unbiased performance estimation on fresh test data.
9. **Business Optimization**: Threshold selection to minimize financial loss.
10. **Explainability (SHAP)**: Global and local feature importance analysis.
11. **Data Drift Monitoring**: Statistical monitoring of feature distributions.
12. **Model Registration**: Centralized registration with business metadata.


In [ ]:
# Configure Plotly to properly render in HTML exports
import plotly.io as pio

# Set the renderer for notebook display
pio.renderers.default = "notebook"

# Configure global theme for consistent appearance
pio.templates.default = "plotly_white"

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

print("✅ Plotly configured for HTML export (renderer='notebook', template='plotly_white')")

In [ ]:
import sys
import os
import importlib
import pandas as pd
import numpy as np
import mlflow
import matplotlib.pyplot as plt
import seaborn as sns

if os.path.exists('/app/src'):
    sys.path.insert(0, '/app/src')
    DATA_PATH = '/app/dataset'
else:
    sys.path.insert(0, os.path.abspath('../src'))
    DATA_PATH = '../dataset'

from classes.data_loader import DataLoader
from classes.sqlite_connector import DatabaseConnection
from classes.feature_engineering import FeatureEngineering
from classes.business_scorer import BusinessScorer
from classes.model_trainer import ModelTrainer
from classes.eda_visualizer import EDAVisualizer
from classes.outlier_analyzer import OutlierAnalyzer
from classes.model_visualizer import ModelVisualizer

mlflow.set_tracking_uri("http://mlflow-dev:5005")
mlflow.set_experiment("HomeCredit_DefaultRisk_Proper_v2")

print(f"Data path: {DATA_PATH}")
print("Setup complete!")

## Step 1: Data Exploration (SQL)
We will load the CSV data into a SQLite database to enable SQL-based exploration.

In [ ]:
loader = DataLoader(DATA_PATH)
db_path = os.path.join(DATA_PATH, 'home_credit.db')

if not os.path.exists(db_path):
    print("Creating SQLite database...")
    loader.create_database(db_path)
else:
    print(f"Database already exists at {db_path}")

db = DatabaseConnection(db_path)
print("Tables:", db.get_table_names())


In [ ]:
# Example SQL Query: Check target distribution in application_train
query_target = """
SELECT TARGET, COUNT(*) as count 
FROM application_train 
GROUP BY TARGET
"""
df_target = db.execute_query(query_target)
print(df_target)

## Step 2: Zero-Leakage Architecture
We split the data immediately after loading to ensure that no information from the validation or test sets leaks into the training process.


In [ ]:
df_train_raw = db.read_table('application_train')

SAMPLE_SIZE = 200000 
if len(df_train_raw) > SAMPLE_SIZE:
    print(f"Sampling dataset to {SAMPLE_SIZE} rows...")
    df_train_raw = df_train_raw.sample(n=SAMPLE_SIZE, random_state=42)

from scripts.data_split import create_data_splits, print_split_summary

X_raw = df_train_raw.drop(columns=['TARGET'])
y_raw = df_train_raw['TARGET']

splits = create_data_splits(X_raw, y_raw, test_size=0.2, random_state=42)

X_train = splits['X_train']
y_train = splits['y_train']
X_val = splits['X_val']
y_val = splits['y_val']
X_test_final = splits['X_test_final']
y_test_final = splits['y_test_final']

print_split_summary(splits)


## Step 3: Feature Engineering
We create new features based on domain knowledge, applying them separately to each split to maintain the zero-leakage principle.


In [ ]:
fe = FeatureEngineering()

# Apply feature engineering to each split separately to avoid leakage
# (Simple row-wise engineering is safe, but we do it split-by-split for best practice)
X_train = fe.simple_feature_engineering(X_train)
X_val = fe.simple_feature_engineering(X_val)
X_test_final = fe.simple_feature_engineering(X_test_final)

print("Feature engineering complete on all splits.")

# 🎯 DEFINE CURATED FEATURE LISTS
TOP_NUMERIC_FEATURES = ['EXT_SOURCE_3', 'EXT_SOURCE_2', 'EXT_SOURCE_1', 'HOUR_APPR_PROCESS_START', 
                        'AMT_REQ_CREDIT_BUREAU_YEAR', 'OWN_CAR_AGE', 'CREDIT_TERM', 'AMT_GOODS_PRICE', 
                        'DAYS_EMPLOYED', 'FLAG_WORK_PHONE', 'AMT_CREDIT', 'OBS_30_CNT_SOCIAL_CIRCLE']

TOP_CATEGORICAL_FEATURES = ['NAME_INCOME_TYPE', 'NAME_FAMILY_STATUS', 'FLAG_OWN_CAR', 'NAME_EDUCATION_TYPE', 
                            'OCCUPATION_TYPE', 'ORGANIZATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 
                            'NAME_TYPE_SUITE', 'CODE_GENDER', 'FLAG_OWN_REALTY']


### Step 3.1: Feature Analysis
Visualize distributions and identify outliers using Plotly to understand the data quality.


In [ ]:
# Feature Analysis: Distribution and Outliers
import gc

# Use a sample of X_train for visualization
df_viz_sample = X_train.sample(n=min(20000, len(X_train)), random_state=42)
analyzer = OutlierAnalyzer(df_viz_sample)

# 1. Overview of Outliers across key numerical features
numeric_cols = [c for c in TOP_NUMERIC_FEATURES if c in df_viz_sample.columns]
all_summaries, all_outlier_info, all_stats_info = analyzer.analyze_outliers(columns=numeric_cols)

print("--- Outlier Summary (Training Set Sample) ---")
analyzer.plot_outlier_summary(all_summaries).show()

# 2. Detailed Distribution Analysis
print("--- Detailed Distribution: External Source 2 (EXT_SOURCE_2) ---")
analyzer.compare_variable_outliers('EXT_SOURCE_2').show()

# Clean up
del analyzer, df_viz_sample, all_summaries, all_outlier_info, all_stats_info
gc.collect()


### Step 3.2: Outlier Treatment
Based on the analysis, we handle outliers by replacing them with `NaN` using Training Set bounds, allowing for robust imputation later.


In [ ]:
print("Applying outlier removal (Z-score ±2) using Training Set bounds...")

analyzer_train = OutlierAnalyzer(X_train)
numeric_cols = [c for c in TOP_NUMERIC_FEATURES if c in X_train.columns]

train_bounds = analyzer_train.get_bounds(method_name="Z-score (±2)", columns=numeric_cols)

X_train = analyzer_train.get_cleaned_dataframe(method_name="Z-score (±2)", columns=numeric_cols, bounds=train_bounds)
X_val = OutlierAnalyzer(X_val).get_cleaned_dataframe(method_name="Z-score (±2)", columns=numeric_cols, bounds=train_bounds)
X_test_final = OutlierAnalyzer(X_test_final).get_cleaned_dataframe(method_name="Z-score (±2)", columns=numeric_cols, bounds=train_bounds)

print("Outliers replaced with NaN across all splits using ONLY Training Set statistics.")


In [ ]:
# Numerical Distributions
numeric_cols = [c for c in TOP_NUMERIC_FEATURES if c in X_train.columns]
EDAVisualizer.plot_numerical_distribution(X_train, columns=numeric_cols)


In [ ]:
# Outlier Analysis with Multiple Methods
import gc

# Use a smaller sample of X_train for visualization
df_sample_outliers = X_train.sample(n=min(5000, len(X_train)), random_state=42)
analyzer = OutlierAnalyzer(df_sample_outliers)
numeric_cols = [c for c in TOP_NUMERIC_FEATURES if c in df_sample_outliers.columns]

# Analyze using all methods
all_summaries, all_outlier_info, all_stats_info = analyzer.analyze_outliers(columns=numeric_cols)

# Plot summary comparison
print("Interactive Outlier Analysis Summary (Training Set):")
analyzer.plot_outlier_summary(all_summaries).show()

# Clean up to save memory
del analyzer, df_sample_outliers, all_summaries, all_outlier_info, all_stats_info
gc.collect()


### Step 3.3: Feature Correlation Analysis
We analyze feature correlations to identify redundant features and understand relationships between variables.


In [ ]:
# Correlation Analysis
from classes.feature_correlation_matrix import CorrelationAnalysis

print("--- Feature Correlation Analysis (Training Set) ---")
# Use a sample of X_train for speed
df_corr_sample = X_train.sample(n=min(20000, len(X_train)), random_state=42)

# Filter for Top Numeric Features
available_features = [c for c in TOP_NUMERIC_FEATURES if c in df_corr_sample.columns]
df_corr_sample = df_corr_sample[available_features]

# Initialize and plot
corr_analyzer = CorrelationAnalysis(df_corr_sample)
fig_corr = corr_analyzer.plot_correlation_matrix()
fig_corr.show()

# Clean up
del corr_analyzer, df_corr_sample
import gc
gc.collect()


## Step 4: Advanced Feature Analysis (PCA & Clustering)
We perform PCA and Clustering analysis to understand the data structure and potential groupings before modeling.


In [ ]:
# Import Analysis Classes
from classes.pca_analysis import PCAAnalysis
from classes.kmean_cluster_analysis import KMeansClusterAnalysis
from classes.dbscan_cluster_analysis import DBSCANClusterAnalysis
import gc
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer

# Use a manageable sample from X_TRAIN for analysis (Zero Leakage)
SAMPLE_SIZE_ANALYSIS = 15000
X_cluster_sample = X_train.sample(n=min(SAMPLE_SIZE_ANALYSIS, len(X_train)), random_state=42)

print(f"Data sampled ({len(X_cluster_sample)} rows) from X_train for Mega Monkey Mode analysis.")


### 4.1 PCA Analysis

In [ ]:
print("--- PCA Analysis ---")

# 1. Select all numeric features from the sample
pca_features = X_cluster_sample.select_dtypes(include=['number']).columns.tolist()
pca_features = [f for f in pca_features if f not in ['TARGET', 'SK_ID_CURR']]

# 2. Normalization
scaler_pca = StandardScaler()
X_scaled = scaler_pca.fit_transform(X_cluster_sample[pca_features])

# 3. KNN Imputation
print(f"Performing KNN Imputation on {len(pca_features)} features...")
imputer_knn = KNNImputer(n_neighbors=5)
X_imputed = imputer_knn.fit_transform(X_scaled)

# Convert back to DataFrame for the analyzer
df_pca_ready = pd.DataFrame(
    X_imputed,
    columns=pca_features,
    index=X_cluster_sample.index
)

# 4. Run PCA
pca_analyzer = PCAAnalysis(df_pca_ready, n_components=10)
pca_components = pca_analyzer.X

# Update df_cluster_sample_filled for K-Means consistency
df_cluster_sample_filled = df_pca_ready

# Plot explained variance
fig_var = pca_analyzer.plot_explained_variance()
fig_var.show()

print(f"PCA completed with {len(pca_features)} features.")


### 4.2 K-Means Clustering Analysis

In [ ]:
# K-Means Analysis
print("--- K-Means Analysis ---")
kmeans_analyzer = KMeansClusterAnalysis(
    df_cluster_sample_filled, 
    pca_components=pca_components
)

# 1. Elbow Method (to find optimal k)
# We check k from 2 to 8
fig_elbow = kmeans_analyzer.plot_elbow(range(2, 9))
fig_elbow.show()

# 2. Fit K-Means (e.g., k=4 based on typical business segments or elbow)
k_selected = 4
print(f"Fitting K-Means with k={k_selected}...")
labels = kmeans_analyzer.fit_kmeans(n_clusters=k_selected)

# 3. Cluster Profile (Feature Importance)
# What features drive the clusters?
fig_feat_imp = kmeans_analyzer.plot_feature_importance(n_clusters=k_selected)
fig_feat_imp.show()

# 4. Intercluster Distance
fig_dist = kmeans_analyzer.plot_intercluster_distance(n_clusters=k_selected)
fig_dist.show()

# 5. Detailed Cluster Profiles (New Method)
print("\n--- Cluster Profiles (Top 10 Features by Variance) ---")
cluster_profiles = kmeans_analyzer.get_cluster_profiles(n_clusters=k_selected)
print(f"\nCluster Summary:")
for cluster_id in range(k_selected):
    cluster_size = cluster_profiles.loc[cluster_id, 'Size']
    cluster_pct = cluster_profiles.loc[cluster_id, 'Pct']
    print(f"  🔷 Cluster {cluster_id}: {int(cluster_size)} samples ({cluster_pct}%)")

display(cluster_profiles.style.background_gradient(cmap='RdYlGn', subset=cluster_profiles.columns[2:]).set_caption('Cluster Profiles (Mean Feature Values)'))


### 4.3 DBSCAN Preparation (K-Distance Graph)

In [ ]:
# DBSCAN Preparation (KNN Distance)
print("--- DBSCAN Prep: K-Distance Graph ---")
dbscan_analyzer = DBSCANClusterAnalysis(
    df_cluster_sample_filled,
    pca_components=pca_components
)

# Plot K-Distance Graph to find optimal eps
# This uses KNN to find distance to kth neighbor
fig_kdist = dbscan_analyzer.find_optimal_eps(min_samples=5, n_neighbors=5)
fig_kdist.show()

# Clean up
import gc
gc.collect()


## Step 5: Preprocessing Pipeline
Prepare data for modeling using an optimized pipeline: Simple Imputation with Missing Indicators and One-Hot Encoding.


In [ ]:
USE_MONKEY_MODE = True

if USE_MONKEY_MODE:
    print("🐒 Monkey Mode Activated: Using ALL available features!")
    cols_to_exclude = ['TARGET', 'SK_ID_CURR']
    feature_cols = [c for c in X_train.columns if c not in cols_to_exclude]
    numeric_features = X_train[feature_cols].select_dtypes(include=['number']).columns.tolist()
    categorical_features = X_train[feature_cols].select_dtypes(include=['object', 'category']).columns.tolist()
else:
    numeric_features = [c for c in TOP_NUMERIC_FEATURES if c in X_train.columns]
    categorical_features = [c for c in TOP_CATEGORICAL_FEATURES if c in X_train.columns]

print(f"Selected {len(numeric_features)} numeric features and {len(categorical_features)} categorical features.")

preprocessor = fe.create_preprocessor(numeric_features, categorical_features, use_knn=False)


## Step 6: Model Strategy
Define the business cost function: **Cost = 10 * FN + 1 * FP**. This asymmetric scoring reflects the high cost of loan defaults.


In [ ]:
business_scorer = BusinessScorer(fn_cost=10, fp_cost=1)
scorer = business_scorer.get_scorer()
print("Business scorer created (FN cost=10, FP cost=1)")

## Step 7: Baseline & Advanced Modeling
We start with a simple Logistic Regression baseline and then move to high-capacity models like LightGBM and Random Forest to capture non-linear risk patterns.


In [ ]:
from sklearn.linear_model import LogisticRegression
from imblearn.pipeline import Pipeline as ImbPipeline

print(f"✅ Training Baseline on {X_train.shape[0]} rows")

pipeline_baseline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

param_grid_baseline = {'classifier__C': [1.0]}
trainer = ModelTrainer(experiment_name="HomeCredit_DefaultRisk_Proper_v2")

baseline_model = trainer.train_and_log(
    pipeline_baseline, param_grid_baseline, X_train, y_train, scorer, 
    run_name="Step6_Baseline_LogReg"
)
print("Baseline model training complete!")


### PhD Baseline Critique
The baseline is now optimized for speed. By switching from `KNNImputer` to `SimpleImputer`, we have reduced the preprocessing overhead significantly. 

**Observations:**
*   **Metric Check**: If the ROC-AUC is below 0.65, the linear model is failing to capture the non-linear relationships in the credit data.
*   **Convergence**: Logistic Regression with 60k rows and many features might struggle to converge if the data is not well-scaled (which we handled in the pipeline).
*   **Next Step**: We move to LightGBM, which handles non-linearity and missing values natively (though we still provide imputed data for consistency).


### Step 7.1: Advanced Model Training (LightGBM & RF)
We use `HalvingGridSearchCV` to tune high-capacity models on the sampled dataset, focusing on aggressive regularization to ensure generalization.


In [ ]:
from lightgbm import LGBMClassifier

pipeline_lgbm = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LGBMClassifier(
        random_state=42, 
        verbose=-1, 
        n_jobs=4, 
        scale_pos_weight=11.4
    ))
])

param_grid_lgbm = {
    'classifier__n_estimators': [2000, 3000],
    'classifier__learning_rate': [0.005, 0.01],
    'classifier__num_leaves': [63, 127],
    'classifier__min_child_samples': [100, 200],
    'classifier__reg_alpha': [0.5, 1.0],
    'classifier__reg_lambda': [5.0, 10.0],
    'classifier__colsample_bytree': [0.6],
    'classifier__subsample': [0.7]
}

print("--- LightGBM ---")
lgbm_model = trainer.train_and_log(
    pipeline_lgbm, param_grid_lgbm, X_train, y_train, scorer, 
    run_name="Step7_LGBM_Ultra_200k",
    factor=3, 
    n_jobs=4
)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

pipeline_rf = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        random_state=42, 
        n_jobs=4, 
        class_weight='balanced'
    ))
])

param_grid_rf = {
    'classifier__n_estimators': [300, 500, 800],
    'classifier__max_depth': [15, 25, None],
    'classifier__min_samples_leaf': [2, 5, 10],
    'classifier__max_features': ['sqrt']
}

print("--- Random Forest ---")
rf_model = trainer.train_and_log(
    pipeline_rf, param_grid_rf, X_train, y_train, scorer, 
    run_name="Step7_RF_Pro_60k",
    step_name="7_model_training_rf",
    factor=4,
    n_jobs=4
)


## Step 8: Model Evaluation & Selection
We evaluate the models using ROC-AUC and Learning Curves to ensure they are not overfitting and have reached statistical equilibrium.


In [ ]:
from scripts.model_evaluation import evaluate_and_select_models, print_evaluation_summary

models = {
    'LightGBM (HalvingSearch)': lgbm_model,
    'Random Forest (HalvingSearch)': rf_model
}

results = evaluate_and_select_models(models, X_val, y_val, X_test_final, y_test_final, business_scorer)

print("\n🏆 Model Leaderboard (Validation Set):\n")
display(results['leaderboard'].style.highlight_min(subset=['Business Cost (Avg)'], color='lightgreen'))

best_model_name = results['best_model_name']
best_model = results['best_model']

print_evaluation_summary(results)

print("Generating Final Visualizations...\n")
visualizer = ModelVisualizer()
visualizer.plot_model_comparison(models, X_val, y_val, business_scorer).show()

print(f"Generating Learning Curves for {best_model_name}...\n")
X_lc = X_train.sample(n=min(40000, len(X_train)), random_state=42)
y_lc = y_train.loc[X_lc.index]

fig_lc = visualizer.plot_learning_curves(
    {best_model_name: best_model.best_estimator_}, 
    X_lc, y_lc, 
    scorer='roc_auc', cv=5
)
fig_lc.show()


## Step 9: Business Cost Optimization
We move beyond standard metrics (AUC) to financial impact. By assigning costs to False Negatives (Defaults) and False Positives (Lost Opportunities), we find the threshold that maximizes profit.


In [ ]:
# Calculate Optimal Threshold
y_proba_test = best_model.predict_proba(X_test_final)[:, 1]
thresholds, costs = business_scorer.get_cost_curve_data(y_test_final, y_proba_test)
optimal_threshold, min_cost = business_scorer.calculate_optimal_threshold(y_test_final, y_proba_test)

print(f"Optimal Threshold: {optimal_threshold:.2f}")
print(f"Minimum Average Cost: {min_cost:.4f}")

# Plot Cost Curve
fig_cost = visualizer.plot_cost_curve(thresholds, costs, optimal_threshold)
fig_cost.show()

# Compare Confusion Matrices (Naive vs Optimal)
y_pred_naive = (y_proba_test >= 0.5).astype(int)
y_pred_optimal = (y_proba_test >= optimal_threshold).astype(int)

print("\n--- Confusion Matrix: Naive Threshold (0.50) ---")
fig_cm_naive = visualizer.plot_confusion_matrix(y_test_final, y_pred_naive)
fig_cm_naive.show()

print(f"\n--- Confusion Matrix: Optimal Threshold ({optimal_threshold:.2f}) ---")
fig_cm_optimal = visualizer.plot_confusion_matrix(y_test_final, y_pred_optimal)
fig_cm_optimal.show()


In [ ]:
# Reload classes to include new Plotly methods
import importlib
import classes.model_visualizer
import classes.business_scorer
importlib.reload(classes.model_visualizer)
importlib.reload(classes.business_scorer)

from classes.model_visualizer import ModelVisualizer
from classes.business_scorer import BusinessScorer

# Re-initialize with existing parameters
visualizer = ModelVisualizer()
business_scorer = BusinessScorer(fn_cost=10, fp_cost=1)
print("Classes reloaded with new Plotly visualization methods.")


## Step 10: Explainability (SHAP)
We use SHAP (SHapley Additive exPlanations) to understand the global and local drivers of credit risk, ensuring the model's decisions are transparent and justifiable.


In [ ]:
print("Computing SHAP values...")
X_shap_sample = X_train[numeric_features + categorical_features].sample(n=200, random_state=42)
shap_data = visualizer.compute_shap_values(best_model, X_shap_sample)

print("Plotting Global Feature Importance...")
fig_summary = visualizer.plot_shap_summary(shap_data)
fig_summary.show()

print("Plotting Local Feature Importance (Sample 0)...")
fig_local = visualizer.plot_shap_local(shap_data, sample_idx=0)
fig_local.show()


## Step 11: Data Drift Monitoring
We compare the training distribution against the test distribution using the Kolmogorov-Smirnov test to detect potential feature drift that could degrade model performance over time.


In [ ]:
from scripts.data_drift import analyze_drift

print("--- Data Drift Analysis ---")

drift_results = analyze_drift(
    reference_data=X_train,
    current_data=X_test_final,
    numeric_features=numeric_features,
    categorical_features=categorical_features
)


## Step 11.5: Champion vs Challenger Comparison
Before registering the new model, we compare it against the current "Champion" model (the one currently in Production) to ensure that the new "Challenger" model provides a significant improvement in business cost reduction.


In [ ]:
import importlib
import scripts.model_comparison
importlib.reload(scripts.model_comparison)

from scripts.model_comparison import compare_with_production

# Define constants for model registration
EXPERIMENT_NAME = "HomeCredit_DefaultRisk_Proper_v2"
RUN_NAME = "Step7_LGBM_Ultra_200k"
MODEL_NAME = "CreditScoring_BestModel"

# Run comparison against current Production model
is_better = compare_with_production(
    model_name=MODEL_NAME,
    X_test=X_test_final,
    y_test=y_test_final,
    challenger_model=best_model.best_estimator_,
    challenger_threshold=optimal_threshold,
    scorer=business_scorer
)

## Step 12: Model Registration
Finally, we register the best model version in the MLflow Model Registry, including the optimized business threshold and cost metadata for production deployment.


### Pre-Registration Quality Assurance
All unit tests for the business scorer, feature engineering, and threshold logic have passed successfully. This ensures the model's logic is robust before registration.


In [ ]:
from scripts.model_registration import register_best_model
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, accuracy_score, classification_report

EXPERIMENT_NAME = "HomeCredit_DefaultRisk_Proper_v2"
RUN_NAME = "Step7_LGBM_Ultra_200k"
MODEL_NAME = "CreditScoring_BestModel"

# Calculate all metrics on TEST set with optimal threshold
y_proba_final = best_model.predict_proba(X_test_final)[:, 1]
y_pred_final = (y_proba_final >= optimal_threshold).astype(int)

# Core metrics
auc_roc = roc_auc_score(y_test_final, y_proba_final)
f1 = f1_score(y_test_final, y_pred_final)
precision = precision_score(y_test_final, y_pred_final)
recall = recall_score(y_test_final, y_pred_final)
accuracy = accuracy_score(y_test_final, y_pred_final)

# Business metrics
from sklearn.metrics import confusion_matrix
tn, fp, fn, tp = confusion_matrix(y_test_final, y_pred_final).ravel()
business_cost_total = (fn * 10) + (fp * 1)
business_cost_avg = business_cost_total / len(y_test_final)

print("=" * 60)
print("📊 MODEL PERFORMANCE SUMMARY (Test Set)")
print("=" * 60)
print(f"\n🎯 Classification Metrics (threshold={optimal_threshold:.2f}):")
print(f"   • AUC-ROC:    {auc_roc:.4f}")
print(f"   • F1-Score:   {f1:.4f}")
print(f"   • Precision:  {precision:.4f}")
print(f"   • Recall:     {recall:.4f}")
print(f"   • Accuracy:   {accuracy:.4f}")

print(f"\n💰 Business Metrics:")
print(f"   • Optimal Threshold:  {optimal_threshold:.2f}")
print(f"   • Business Cost/Sample: {business_cost_avg:.4f}")
print(f"   • Total Business Cost:  {business_cost_total:,}")
print(f"   • True Positives:  {tp:,} (correctly rejected high-risk)")
print(f"   • True Negatives:  {tn:,} (correctly approved low-risk)")
print(f"   • False Positives: {fp:,} (lost opportunities, cost={fp})")
print(f"   • False Negatives: {fn:,} (defaults, cost={fn*10})")

print(f"\n📋 Classification Report:")
print(classification_report(y_test_final, y_pred_final, target_names=['No Default', 'Default']))

# Store metrics for registration
model_metrics = {
    'auc_roc': auc_roc,
    'f1_score': f1,
    'precision': precision,
    'recall': recall,
    'accuracy': accuracy,
    'optimal_threshold': optimal_threshold,
    'business_cost_avg': business_cost_avg,
    'business_cost_total': int(business_cost_total),
    'true_positives': int(tp),
    'true_negatives': int(tn),
    'false_positives': int(fp),
    'false_negatives': int(fn),
    'test_set_size': len(y_test_final),
    'positive_rate': float(y_test_final.mean())
}

print("=" * 60)

# Only register if the model is better than the current production model (or if it's the first one)
if 'is_better' not in locals() or is_better:
    registered_model = register_best_model(
        experiment_name=EXPERIMENT_NAME,
        run_name=RUN_NAME,
        model_name=MODEL_NAME,
        optimal_threshold=optimal_threshold,
        min_cost=min_cost,
        extra_metrics=model_metrics,  # Pass all metrics
        transition_to_prod=True
    )
    print("\n✅ Model registered with full metrics!")
else:
    print("\n❌ Registration skipped: Challenger did not outperform Champion.")

In [ ]:
import importlib
import scripts.export_model
importlib.reload(scripts.export_model)

from scripts.export_model import export_production_model, verify_exported_model

# Export Production model to prod_models/
export_metadata = export_production_model(
    model_name=MODEL_NAME,
    output_dir="/app/prod_models"
)

# Verify the export
if export_metadata:
    verify_exported_model("/app/prod_models")
    print("\n✅ Model ready for CI/CD deployment!")
    print("Next steps:")
    print("  1. git add prod_models/")
    print("  2. git commit -m 'Update production model'")
    print("  3. git push origin main")
    print("  4. CI/CD pipeline will deploy to Lightsail")

## Step 13: Export Evidently Data Drift Report
Generate and export the Evidently data drift report to `prod_models/` for regulatory compliance and CI/CD deployment.

In [ ]:
# Generate Evidently Data Drift Report
import subprocess
import json
import os
from datetime import datetime

print("📊 Generating Evidently Data Drift Report...")

# Export data for subprocess
X_train[numeric_features + categorical_features].to_parquet('/tmp/reference_data.parquet')
X_test_final[numeric_features + categorical_features].to_parquet('/tmp/current_data.parquet')

# Evidently v0.7+ script
script = '''
import pandas as pd
from evidently import Report
from evidently.presets import DataDriftPreset

ref = pd.read_parquet('/tmp/reference_data.parquet')
cur = pd.read_parquet('/tmp/current_data.parquet')

snapshot = Report([DataDriftPreset()]).run(current_data=cur, reference_data=ref)
snapshot.save_html('/app/prod_models/evidently_data_drift_report.html')
snapshot.save_json('/app/prod_models/evidently_data_drift_report.json')
print("OK")
'''

result = subprocess.run(['python', '-c', script], capture_output=True, text=True)

if result.returncode == 0:
    # Update metadata
    metadata_path = "/app/prod_models/metadata.json"
    if os.path.exists(metadata_path):
        with open(metadata_path, 'r') as f:
            metadata = json.load(f)
        metadata['evidently_report'] = {
            'html': 'evidently_data_drift_report.html',
            'json': 'evidently_data_drift_report.json',
            'generated_at': datetime.now().isoformat()
        }
        with open(metadata_path, 'w') as f:
            json.dump(metadata, f, indent=2)
    
    print("✅ Evidently report exported to prod_models/")
    print("\n📦 prod_models/ contents:")
    for f in sorted(os.listdir("/app/prod_models")):
        print(f"   - {f}")
else:
    print(f"❌ Failed: {result.stderr}")